# Turkish Legal RAG - Qwen LoRA Fine-tuning

Runtime: GPU (T4 or better). This trains Qwen2.5-0.5B-Instruct on the legal RAG SFT data, merges LoRA into the base model, and pushes the fine-tuned model to Hugging Face Hub.

In [ ]:
!nvidia-smi
!pip -q install -U transformers datasets accelerate peft bitsandbytes huggingface_hub safetensors

## 1. Login to Hugging Face
Create a write token at https://huggingface.co/settings/tokens and paste it when asked.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 2. Clone project branch

In [ ]:
!rm -rf NLP-Term-Project
!git clone -b custom-submission-support-2026-05-25 https://github.com/Bloheujkleh/NLP-Term-Project.git
%cd NLP-Term-Project
!ls -lh data/llm.jsonl colab_train_qwen_lora.py

## 3. Configure output model repo
Change `HF_REPO_ID` to your own Hugging Face username/model name.

In [ ]:
import os
os.environ['BASE_MODEL'] = 'Qwen/Qwen2.5-0.5B-Instruct'
os.environ['TRAIN_FILE'] = 'data/llm.jsonl'
os.environ['HF_REPO_ID'] = 'CHANGE_ME/turkish-legal-qwen2-5-0-5b-rag-sft'
os.environ['MAX_EXAMPLES'] = '12000'  # lower to 4000 for quick smoke; 12000 for stronger final
os.environ['MAX_LENGTH'] = '1536'
print(os.environ['HF_REPO_ID'])

## 4. Train + merge + push
This may take around 1-3 hours on T4 depending on MAX_EXAMPLES.

In [ ]:
!python colab_train_qwen_lora.py

## 5. Test the pushed fine-tuned model quickly

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, os
model_id = os.environ['HF_REPO_ID']
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='auto')
messages = [
    {'role':'system','content':'Kaynak disina cikmayan Turkce RAG asistani.'},
    {'role':'user','content':'KAYNAK: Kasten oldurme sucu isleyen kisi muebbet hapis cezasi ile cezalandirilir.\nSORU: Kasten oldurmenin cezasi nedir?\nCEVAP:'}
]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors='pt').to(model.device)
out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
print(tok.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))

## 6. Final Space setting
After training, set the Hugging Face Space environment variable:

`GENERATION_MODEL=<your pushed model id>`

The app will then use the fine-tuned Qwen model instead of the base Qwen model.